# 🦀 Day 7: Mini-Project — Full-Stack URL Shortener

---

## Project Overview

Build a **full-stack URL shortener**: web API + SQLite database + ready for Docker.

- **Axum** — HTTP server and routing
- **rusqlite** — SQLite for persistence
- **serde** — JSON serialization
- **Short codes** — random alphanumeric (e.g., `aB3xY9`)
- **Click tracking** — count redirects

## Architecture Diagram

```
  Client                    Axum Server                    SQLite
    │                            │                            │
    │  POST /shorten             │   INSERT short_url         │
    │  { "url": "..." }  ──────►│   generate code    ──────►│
    │                            │                            │
    │  GET /:code                │   SELECT original_url      │
    │  (redirect)         ◄──────│   UPDATE clicks     ◄──────│
    │                            │                            │
    │  GET /stats/:code          │   SELECT clicks, etc       │
    │  { "clicks": 42 }   ◄──────│   ───────────────────────►│
```

## Data Model

```rust
struct ShortUrl {
    id: i64,
    short_code: String,
    original_url: String,
    created_at: String,
    clicks: i64,
}
```

## Dependencies for Cargo.toml

```toml
[dependencies]
axum = { version = "0.7", features = ["json"] }
tokio = { version = "1", features = ["full"] }
rusqlite = { version = "0.31", features = ["bundled"] }
serde = { version = "1.0", features = ["derive"] }
serde_json = "1.0"
tower-http = { version = "0.5", features = ["cors"] }
rand = "0.8"
chrono = "0.4"
```

## Routes

| Method | Path | Description |
|--------|------|-------------|
| POST | `/shorten` | Create short URL, body: `{"url": "https://..."}` |
| GET | `/:code` | Redirect to original URL, increment clicks |
| GET | `/stats/:code` | Return `{"short_code", "original_url", "clicks", "created_at"}` |

## Short Code Generation

Random alphanumeric string (e.g., 6 chars). Collision handling: retry or lengthen.

In [ ]:
:dep rand = "0.8"

use rand::Rng;

const CHARSET: &[u8] = b"abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789";

fn generate_short_code(len: usize) -> String {
    let mut rng = rand::thread_rng();
    (0..len)
        .map(|_| {
            let idx = rng.gen_range(0..CHARSET.len());
            CHARSET[idx] as char
        })
        .collect()
}

println!("Sample codes: {} {} {}", generate_short_code(6), generate_short_code(6), generate_short_code(6));

## Full Axum Application (Cargo Binary)

This code is meant for a Cargo project. It won't run in EvCxR (needs `main`, server, file I/O). Structure:

### Project File Structure

```
url-shortener/
├── Cargo.toml
├── src/
│   └── main.rs
├── Dockerfile
├── docker-compose.yml
└── .dockerignore
```

In [ ]:
// === src/main.rs (complete, for Cargo project) ===
// Run with: cargo run

/*
use axum::{
    extract::{Path, State},
    http::StatusCode,
    response::Redirect,
    routing::{get, post},
    Json, Router,
};
use rusqlite::{params, Connection};
use serde::{Deserialize, Serialize};
use std::sync::Mutex;
use rand::Rng;

const CHARSET: &[u8] = b"abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789";

#[derive(Clone)]
struct AppState {
    db: Mutex<Connection>,
}

#[derive(Deserialize)]
struct ShortenRequest {
    url: String,
}

#[derive(Serialize)]
struct ShortenResponse {
    short_code: String,
    short_url: String,
}

#[derive(Serialize)]
struct StatsResponse {
    short_code: String,
    original_url: String,
    clicks: i64,
    created_at: String,
}

fn generate_code(len: usize) -> String {
    let mut rng = rand::thread_rng();
    (0..len).map(|_| CHARSET[rng.gen_range(0..CHARSET.len())] as char).collect()
}

fn init_db(conn: &Connection) -> rusqlite::Result<()> {
    conn.execute(
        "CREATE TABLE IF NOT EXISTS short_urls (
            id INTEGER PRIMARY KEY,
            short_code TEXT UNIQUE NOT NULL,
            original_url TEXT NOT NULL,
            created_at TEXT NOT NULL,
            clicks INTEGER DEFAULT 0
        )",
        [],
    )?;
    Ok(())
}

async fn shorten(State(state): State<AppState>, Json(req): Json<ShortenRequest>) -> Result<Json<ShortenResponse>, StatusCode> {
    let code = loop {
        let c = generate_code(6);
        let db = state.db.lock().unwrap();
        let exists: bool = db.query_row("SELECT 1 FROM short_urls WHERE short_code = ?1", [&c], |_| Ok(())).is_ok();
        if !exists { break c; }
    };
    let now = chrono::Utc::now().format("%Y-%m-%d %H:%M:%S").to_string();
    let db = state.db.lock().unwrap();
    db.execute("INSERT INTO short_urls (short_code, original_url, created_at) VALUES (?1, ?2, ?3)", params![&code, &req.url, &now]).map_err(|_| StatusCode::INTERNAL_SERVER_ERROR)?;
    Ok(Json(ShortenResponse { short_code: code.clone(), short_url: format!("/{}", code) }))
}

async fn redirect(State(state): State<AppState>, Path(code): Path<String>) -> Result<Redirect, StatusCode> {
    let (url,): (String,) = state.db.lock().unwrap().query_row(
        "SELECT original_url FROM short_urls WHERE short_code = ?1", [&code], |r| Ok((r.get(0)?,))
    ).map_err(|_| StatusCode::NOT_FOUND)?;
    state.db.lock().unwrap().execute("UPDATE short_urls SET clicks = clicks + 1 WHERE short_code = ?1", [&code]).ok();
    Ok(Redirect::temporary(&url))
}

async fn stats(State(state): State<AppState>, Path(code): Path<String>) -> Result<Json<StatsResponse>, StatusCode> {
    let row = state.db.lock().unwrap().query_row(
        "SELECT original_url, clicks, created_at FROM short_urls WHERE short_code = ?1", [&code], |r| Ok((r.get::<_, String>(0)?, r.get::<_, i64>(1)?, r.get::<_, String>(2)?))
    ).map_err(|_| StatusCode::NOT_FOUND)?;
    Ok(Json(StatsResponse { short_code: code, original_url: row.0, clicks: row.1, created_at: row.2 }))
}

#[tokio::main]
async fn main() {
    let conn = Connection::open("./urls.db").expect("open db");
    init_db(&conn).expect("init db");
    let state = AppState { db: Mutex::new(conn) };
    let app = Router::new()
        .route("/shorten", post(shorten))
        .route("/:code", get(redirect))
        .route("/stats/:code", get(stats))
        .with_state(state);
    let listener = tokio::net::TcpListener::bind("0.0.0.0:3000").await.unwrap();
    axum::serve(listener, app).await.unwrap();
}
*/

// Above is the full app. Copy to src/main.rs and run with cargo run.
println!("See full code in cell — copy to src/main.rs for Cargo project");

## Error Handling

- **404** — short code not found
- **500** — DB errors, insert failures
- Use `Result<Json<T>, StatusCode>` or custom error type with `IntoResponse`

## Dockerfile and docker-compose.yml

**Dockerfile** (multi-stage):

```dockerfile
FROM rust:1.75-slim as builder
WORKDIR /app
COPY . .
RUN cargo build --release

FROM debian:bookworm-slim
RUN apt-get update && apt-get install -y ca-certificates && rm -rf /var/lib/apt/lists/*
WORKDIR /app
COPY --from=builder /app/target/release/url-shortener .
EXPOSE 3000
CMD ["./url-shortener"]
```

**docker-compose.yml** (app + SQLite volume):

```yaml
version: "3.8"
services:
  app:
    build: .
    ports:
      - "3000:3000"
    volumes:
      - ./data:/app
    environment:
      - DATABASE_PATH=/app/urls.db
```

## API Usage Examples (curl)

```bash
# Create short URL
curl -X POST http://localhost:3000/shorten -H "Content-Type: application/json" -d '{"url":"https://rust-lang.org"}'
# {"short_code":"aB3xY9","short_url":"/aB3xY9"}

# Redirect (follow in browser or curl -L)
curl -L http://localhost:3000/aB3xY9

# Get stats
curl http://localhost:3000/stats/aB3xY9
# {"short_code":"aB3xY9","original_url":"https://rust-lang.org","clicks":5,"created_at":"2025-03-08 12:00:00"}
```

## 🏋️ Exercise

Extend the URL shortener with one or more of:
- **URL validation** — reject invalid URLs (use `url` crate or regex)
- **Expiration** — add `expires_at` column, skip expired in redirect
- **Custom short codes** — allow `POST /shorten` with optional `{"url":"...", "code":"myCustom"}`

Implement your chosen feature in the data model and handlers.

In [ ]:
// YOUR CODE HERE
// Example: URL validation helper

fn is_valid_url(s: &str) -> bool {
    s.starts_with("http://") || s.starts_with("https://")
}

// Extend with expiration or custom codes as you prefer!
println!("is_valid_url(https://example.com) = {}", is_valid_url("https://example.com"));
println!("is_valid_url(ftp://bad) = {}", is_valid_url("ftp://bad"));

## 🎯 Key Takeaways

1. **Full-stack Rust** — Axum + rusqlite = web API + persistence
2. **Data model** — ShortUrl with short_code, original_url, clicks, created_at
3. **Routes** — POST /shorten, GET /:code (redirect), GET /stats/:code
4. **Short codes** — random alphanumeric; handle collisions
5. **Click tracking** — UPDATE clicks on redirect
6. **Docker** — multi-stage build + compose for deployment

---

## 🎉 Month 4 Complete!

**Next:** Month 5 — Systems Programming & Advanced Topics! 🚀